<a href="https://colab.research.google.com/github/Iam0-0ap/Latent-Space-Visualization/blob/main/Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [2]:
class ReverseModel(nn.Module):

    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )

    def forward(self, x_t, t):

        t = t.float().unsqueeze(1)

        x = torch.cat([x_t, t], dim=1)

        mu_theta = self.net(x)

        return mu_theta

In [3]:
# x_t = sqrt(1-β_t) x_{t-1} + sqrt(β_t) ε

In [4]:
class Diffusion:

    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):

        self.T = T
        self.device = device

        self.beta = torch.linspace(beta_start, beta_end, T).to(device)

    def forward_step(self, x_prev, t):

        beta_t = self.beta[t]

        noise = torch.randn_like(x_prev)

        x_t = torch.sqrt(1 - beta_t) * x_prev + torch.sqrt(beta_t) * noise

        return x_t

In [5]:
def diffuse_trajectory(diffusion, x0):

    x = x0
    trajectory = [x0]

    for t in range(diffusion.T):

        x = diffusion.forward_step(x, t)

        trajectory.append(x)

    return trajectory

In [6]:
def train_step(model, diffusion, optimizer, x0):

    device = x0.device
    B = x0.shape[0]

    trajectory = diffuse_trajectory(diffusion, x0)

    loss = 0

    for t in reversed(range(1, diffusion.T)):

        x_t = trajectory[t]
        x_prev = trajectory[t-1]

        t_tensor = torch.full((B,), t, device=device)

        mu_theta = model(x_t, t_tensor)

        loss += F.mse_loss(mu_theta, x_prev)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [7]:
# reverse diffusion
@torch.no_grad()
def sample(model, diffusion, shape):

    device = diffusion.device

    x = torch.randn(shape).to(device)

    for t in reversed(range(diffusion.T)):

        t_tensor = torch.full((shape[0],), t, device=device)

        mu_theta = model(x, t_tensor)

        beta_t = diffusion.beta[t]

        if t > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = mu_theta + torch.sqrt(beta_t) * noise

    return x

In [19]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# Define epochs
epochs = 100

# Initialize DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [ ]:



model = ReverseModel(dim=784).to(device)

diffusion = Diffusion(T=1000, device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(epochs):

    for x,_ in dataloader:

        x = x.view(x.size(0), -1).to(device)

        loss = train_step(model, diffusion, optimizer, x)

    print("epoch", epoch, "loss", loss)

epoch 0 loss 711.7901000976562
epoch 1 loss 644.6569213867188
epoch 2 loss 605.833251953125
epoch 3 loss 586.1033325195312
epoch 4 loss 567.0888061523438
epoch 5 loss 556.6810302734375
epoch 6 loss 548.1271362304688
epoch 7 loss 543.3863525390625
epoch 8 loss 537.9361572265625
epoch 9 loss 535.533935546875
epoch 10 loss 536.273193359375
epoch 11 loss 533.04443359375
epoch 12 loss 533.5310668945312
epoch 13 loss 529.9260864257812
epoch 14 loss 530.4526977539062
epoch 15 loss 532.0675659179688
epoch 16 loss 531.5638427734375
epoch 17 loss 531.8203125
epoch 18 loss 530.8727416992188
epoch 19 loss 532.3870849609375
epoch 20 loss 528.7011108398438
epoch 21 loss 529.9553833007812
epoch 22 loss 527.19140625
epoch 23 loss 528.1372680664062
epoch 24 loss 528.1703491210938
epoch 25 loss 530.3198852539062
epoch 26 loss 530.9676513671875
epoch 27 loss 531.1978149414062
epoch 28 loss 524.6632080078125
epoch 29 loss 529.1585083007812
epoch 30 loss 530.7757568359375
epoch 31 loss 527.5823974609375
ep

In [1]:
samples = sample(model, diffusion, (16, 784))

samples = samples.view(16, 1, 28, 28)

NameError: name 'sample' is not defined

In [ ]:
samples

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(samples[0][0].detach().cpu(), cmap="gray")
plt.show()